# 🔗 MeTHash — Pipeline de Détection d'URLs Malveillantes

Ce notebook implémente un pipeline complet de Machine Learning pour classifier les URLs en 4 catégories :
- **Bénigne** (sûre)
- **Phishing** (tentative d'hameçonnage)
- **Malware** (logiciel malveillant)
- **Defacement** (site web défiguré)

## Étapes du pipeline :
1. Import des dépendances
2. Chargement et exploration des données
3. Encodage des labels
4. Ingénierie des caractéristiques (Feature Engineering)
5. Division train/test
6. Gestion du déséquilibre avec SMOTE
7. Entraînement d'un Random Forest
8. Évaluation du modèle
9. Sauvegarde du modèle

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib
import re
import warnings
from urllib.parse import urlparse
from pathlib import Path

# ML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
import joblib

# Feature extraction
import tldextract
import validators

warnings.filterwarnings('ignore')

# Plot style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Toutes les bibliothèques importées avec succès !")
print(f"   pandas: {pd.__version__}")
print(f"   numpy:  {np.__version__}")
print(f"   sklearn: {__import__('sklearn').__version__}")

## 2. Chargement et exploration du dataset

In [ ]:
# ── Configuration des chemins ───────────────────────────────────────────
DATA_PATH = Path.cwd().parent / 'data' / 'malicious_phish.csv'
MODEL_DIR = Path.cwd().parent / 'models'
MODEL_DIR.mkdir(exist_ok=True)

# ── Chargement ──────────────────────────────────────────────────────────
print(f"[*] Chargement depuis : {DATA_PATH}")
df = pd.read_csv(DATA_PATH)
print(f"[✓] Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# ── Aperçu ──────────────────────────────────────────────────────────────
df.head(10)

In [ ]:
# ── Exploration des données ─────────────────────────────────────────────
print("📋 Informations générales :")
df.info()

print("\n📊 Statistiques descriptives :")
df.describe(include='all')

In [ ]:
# ── Détection automatique des colonnes ──────────────────────────────────
possible_label_cols = ['type', 'label', 'category', 'class', 'Type', 'Label']
label_col = None
for col in possible_label_cols:
    if col in df.columns:
        label_col = col
        break

if label_col is None:
    label_col = df.columns[-1]
    
possible_url_cols = ['url', 'URL', 'Url', 'domain']
url_col = None
for col in possible_url_cols:
    if col in df.columns:
        url_col = col
        break

print(f"Colonne URL détectée : '{url_col}'")
print(f"Colonne label détectée : '{label_col}'")

# ── Distribution des classes ────────────────────────────────────────────
class_counts = df[label_col].value_counts()
print("\n📊 Distribution des classes :")
print(class_counts)

# ── Visualisation ───────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
colors = ['#00c853', '#ff1744', '#ff9100', '#ffd600']
bars = plt.bar(class_counts.index, class_counts.values, color=colors[:len(class_counts)])
plt.title('Distribution des classes dans le dataset', fontsize=16, fontweight='bold')
plt.xlabel('Catégorie', fontsize=12)
plt.ylabel('Nombre d\'URLs', fontsize=12)
plt.xticks(rotation=45)

# Ajouter les valeurs sur les barres
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height):,}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\nℹ️  Nombre total d'URLs : {len(df):,}")
print(f"ℹ️  Nombre de classes : {len(class_counts)}")
print(f"ℹ️  Classes : {list(class_counts.index)}")

## 3. Encodage des labels

In [ ]:
# ── Renommer les colonnes pour standardisation ─────────────────────────
df.rename(columns={label_col: 'type', url_col: 'url'}, inplace=True)

# ── Normaliser et encoder ──────────────────────────────────────────────
df['type'] = df['type'].str.lower().str.strip()

label_encoder = LabelEncoder()
df['type_encoded'] = label_encoder.fit_transform(df['type'])

# Afficher le mapping
print("📋 Mapping des labels :")
for i, label in enumerate(label_encoder.classes_):
    print(f"   {i} → {label}")

# Vérification
print("\n✅ Aperçu après encodage :")
df[['url', 'type', 'type_encoded']].head(10)

## 4. Ingénierie des caractéristiques (Feature Engineering)

In [ ]:
# ── Liste des raccourcisseurs d'URL ────────────────────────────────────
SHORTENERS = {
    'bit.ly', 'tinyurl.com', 'goo.gl', 'ow.ly', 't.co', 'is.gd', 'buff.ly',
    'shorturl.at', 'rb.gy', 'short.link', 'cutt.ly', 'rebrand.ly',
    'soo.gd', 's2r.co', 'bl.ink', 'v.gd', '2.gp', 'tiny.cc', 'tr.im',
    'clck.ru', '0x0.st', 's.id', 'shortcm.xyz', 'x.co', 'qr.ae',
}

# ── Mots-clés sensibles ─────────────────────────────────────────────────
SENSITIVE_WORDS = ['login', 'verify', 'secure', 'update', 'bank', 'account',
                   'confirm', 'signin', 'webscr', 'password', 'credential']

# ── TLDs suspects ───────────────────────────────────────────────────────
SUSPICIOUS_TLDS = {'tk', 'ml', 'ga', 'cf', 'gq', 'xyz', 'top', 'work', 'loan'}


def is_ip_address(hostname):
    """Vérifie si le hostname est une adresse IP."""
    try:
        if validators.ipv4(hostname) or validators.ipv6(hostname):
            return 1
    except Exception:
        pass
    ipv4_pattern = r'^(\d{1,3}\.){3}\d{1,3}$'
    if re.match(ipv4_pattern, hostname):
        parts = hostname.split('.')
        if all(0 <= int(p) <= 255 for p in parts):
            return 1
    return 0


def shannon_entropy(text):
    """Calcule l'entropie de Shannon (mesure de caractères aléatoires)."""
    if not text:
        return 0.0
    text = str(text)
    prob = [text.count(c) / len(text) for c in set(text)]
    return -sum(p * np.log2(p) for p in prob if p > 0)


def extract_features(url):
    """Extrait les caractéristiques numériques d'une URL."""
    url_str = str(url).strip()
    parsed = urlparse(url_str)
    hostname = parsed.hostname or ''
    path = parsed.path or ''
    query = parsed.query or ''
    ext = tldextract.extract(url_str)
    domain = ext.domain
    suffix = ext.suffix
    subdomain = ext.subdomain
    full_domain = f"{domain}.{suffix}" if suffix else domain

    features = {
        'url_len': len(url_str),
        'digits_count': sum(c.isdigit() for c in url_str),
        'special_chars_count': sum(1 for c in url_str if c in "!@#$%^&*()_+-=[]{}|;':\",./<>?~`"),
        'secure_http': 1 if parsed.scheme == 'https' else 0,
        'shortened': 1 if hostname.lower() in SHORTENERS else 0,
        'have_ip': is_ip_address(hostname),
        'nb_dots': url_str.count('.'),
        'nb_hyphens': url_str.count('-'),
        'nb_slash': url_str.count('/'),
        'nb_question_mark': url_str.count('?'),
        'nb_eq': url_str.count('='),
        'nb_at': url_str.count('@'),
        'nb_and': url_str.count('&'),
        'nb_or': url_str.count('|'),
        'nb_hash': url_str.count('#'),
        'nb_percent': url_str.count('%'),
        'url_depth': path.count('/'),
        'hostname_len': len(hostname),
        'path_len': len(path),
        'query_len': len(query),
        'is_encoded': 1 if re.search(r'%[0-9a-fA-F]{2}', url_str) else 0,
        'tld_length': len(suffix),
        'subdomain_len': len(subdomain),
        'domain_len': len(domain),
        'domain_entropy': shannon_entropy(domain),
        'nb_subdomains': len([s for s in subdomain.split('.') if s]) if subdomain else 0,
        'suspicious_tld': 1 if suffix in SUSPICIOUS_TLDS else 0,
        'sensitive_words': sum(1 for w in SENSITIVE_WORDS if w in url_str.lower()),
        'root_domain_hash': int(hashlib.md5(full_domain.encode()).hexdigest(), 16) % (10 ** 8),
    }
    return features


# ── Appliquer l'extraction à toutes les URLs ────────────────────────────
print("[*] Extraction des caractéristiques pour toutes les URLs...")
feature_records = df['url'].apply(extract_features)
X = pd.DataFrame(feature_records.tolist())
print(f"[✓] {X.shape[1]} caractéristiques extraites de {X.shape[0]} URLs.")

# Afficher les noms des features
print("\n📋 Liste des caractéristiques :")
for i, col in enumerate(X.columns, 1):
    print(f"   {i:>2}. {col}")

X.head()

## 5. Division des données (Train / Test)

In [ ]:
# ── Préparation des matrices ───────────────────────────────────────────
y = df['type_encoded'].values
feature_names = list(X.columns)

# ── Division 70/30 avec stratification ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"📊 Dimensions des ensembles :")
print(f"   X_train : {X_train.shape}")
print(f"   X_test  : {X_test.shape}")
print(f"   y_train : {y_train.shape}")
print(f"   y_test  : {y_test.shape}")

# Distribution des classes dans chaque ensemble
print(f"\n📊 Distribution des classes (entraînement) :")
train_dist = pd.Series(y_train).value_counts().sort_index()
for i, count in train_dist.items():
    print(f"   Classe {i} ({label_encoder.classes_[i]}) : {count}")

print(f"\n📊 Distribution des classes (test) :")
test_dist = pd.Series(y_test).value_counts().sort_index()
for i, count in test_dist.items():
    print(f"   Classe {i} ({label_encoder.classes_[i]}) : {count}")

## 6. Gestion du déséquilibre avec SMOTE

In [ ]:
print("[*] Application de SMOTE pour équilibrer les classes...")
print(f"   Avant SMOTE - Distribution : {np.bincount(y_train)}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"   Après SMOTE - Distribution : {np.bincount(y_train_resampled)}")
print(f"   Taille de l'ensemble d'entraînement : {X_train.shape[0]} → {X_train_resampled.shape[0]}")
print("✅ SMOTE appliqué avec succès !")

## 7. Entraînement du modèle Random Forest

In [ ]:
print("[*] Initialisation du Random Forest Classifier...")
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
    verbose=1,
)

print("[*] Entraînement en cours (cela peut prendre quelques minutes)...")
model.fit(X_train_resampled, y_train_resampled)
print("✅ Modèle entraîné avec succès !")

## 8. Évaluation du modèle

In [ ]:
# ── Prédictions sur l'ensemble de test ─────────────────────────────────
print("[*] Évaluation sur l'ensemble de test...")
y_pred = model.predict(X_test.values)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n{'='*60}")
print(f"🎯 Accuracy : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*60}")

# ── Rapport de classification ───────────────────────────────────────────
print(f"\n📋 Classification Report :")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# ── Matrice de confusion ───────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Matrice de Confusion', fontsize=16, fontweight='bold')
plt.ylabel('Vraie classe', fontsize=12)
plt.xlabel('Classe prédite', fontsize=12)
plt.tight_layout()
plt.show()

# ── Feature Importance ──────────────────────────────────────────────────
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 20))
bars = plt.barh(range(20), importances[indices[:20]][::-1], color=colors[::-1])
plt.yticks(range(20), [feature_names[i] for i in indices[:20]][::-1])
plt.title('Top 20 des caractéristiques les plus importantes', fontsize=16, fontweight='bold')
plt.xlabel('Importance', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Top 15 des caractéristiques :")
for i, idx in enumerate(indices[:15], 1):
    print(f"   {i:>2}. {feature_names[idx]:<25s} → {importances[idx]:.4f}")

## 9. Sauvegarde du modèle

In [ ]:
# ── Sauvegarde du modèle et des métadonnées ────────────────────────────
model_path = MODEL_DIR / 'url_model.sav'
features_path = MODEL_DIR / 'feature_names.joblib'
encoder_path = MODEL_DIR / 'label_encoder.joblib'
metadata_path = MODEL_DIR / 'metadata.joblib'

joblib.dump(model, model_path)
joblib.dump(feature_names, features_path)
joblib.dump(label_encoder, encoder_path)

# Métadonnées
metadata = {
    'accuracy': accuracy,
    'feature_count': len(feature_names),
    'features': feature_names,
    'model_type': 'RandomForest',
    'n_estimators': 200,
    'max_depth': 30,
    'classes': list(label_encoder.classes_),
    'training_samples': len(X_train_resampled),
    'test_samples': len(X_test),
}
joblib.dump(metadata, metadata_path)

print(f"✅ Modèle sauvegardé dans : {model_path}")
print(f"✅ Features sauvegardées dans : {features_path}")
print(f"✅ Label encoder sauvegardé dans : {encoder_path}")
print(f"✅ Métadonnées sauvegardées dans : {metadata_path}")

# Taille des fichiers
import os
for path in [model_path, features_path, encoder_path, metadata_path]:
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"   📦 {path.name} : {size_mb:.2f} MB")

print(f"\n{'='*60}")
print(f"🎯 PIPELINE TERMINÉ AVEC SUCCÈS !")
print(f"   Accuracy finale : {accuracy:.2%}")
print(f"{'='*60}")